In [1]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.5 MB/s eta 0:00:00


In [2]:
import json
import pickle
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

INPUT_DIR    = Path("/kaggle/input/notebooks/lehuuluongb2306557/faiss-index/index_artifacts")
OUTPUT_DIR   = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiem tra dataset da duoc mount chua
assert INPUT_DIR.exists(), (
    f"Khong tim thay {INPUT_DIR}. "
)

# Xac nhan cac file bat buoc ton tai
REQUIRED = [
    "interactions.csv",
    "metadata_indexed.csv",
    "combined_embeddings.npy",
    "item_ids.npy",
    "fashion.index",
    "index_config.json",
]

for fname in REQUIRED:
    fpath = INPUT_DIR / fname
    assert fpath.exists(), f"Thieu file bat buoc: {fpath}"
    print(f"  OK  {fname}  ({fpath.stat().st_size / 1e6:.1f} MB)")

print("\nTat ca file dau vao hop le.")

  OK  interactions.csv  (0.3 MB)
  OK  metadata_indexed.csv  (13.1 MB)
  OK  combined_embeddings.npy  (134.9 MB)
  OK  item_ids.npy  (0.5 MB)
  OK  fashion.index  (134.9 MB)
  OK  index_config.json  (0.0 MB)

Tat ca file dau vao hop le.


In [3]:
df_inter = pd.read_csv(INPUT_DIR / "interactions.csv")

# Ep kieu du lieu
df_inter["user_id"] = df_inter["user_id"].astype(int)
df_inter["item_id"] = df_inter["item_id"].astype(int)
df_inter["rating"]  = df_inter["rating"].astype(float)

print(f"interactions.csv: {len(df_inter):,} dong")
print(f"  user_id: {df_inter['user_id'].nunique():,} users unik")
print(f"  item_id: {df_inter['item_id'].nunique():,} items unik")
print(f"  rating : [{df_inter['rating'].min()}, {df_inter['rating'].max()}]")
df_inter.head()

interactions.csv: 29,785 dong
  user_id: 1,000 users unik
  item_id: 23,929 items unik
  rating : [1.0, 5.0]


,user_id,item_id,rating
0,0,14592,4.0
1,0,3278,4.0
2,0,36048,5.0
3,0,32098,3.0
4,0,29256,4.0


In [4]:
# Xay anh xa user_id (goc) --> chi so hang (0-based) va nguoc lai
unique_users = sorted(df_inter["user_id"].unique())
unique_items = sorted(df_inter["item_id"].unique())

# key la str vi JSON chi ho tro key la string
user_mapping = {int(u): idx for idx, u in enumerate(unique_users)}  # user_id -> row
item_mapping = {int(i): idx for idx, i in enumerate(unique_items)}  # item_id -> col

# Mapping nguoc: chi so -> id goc (dung khi giai ma ket qua CF)
inv_user_mapping = {v: k for k, v in user_mapping.items()}  # row  -> user_id
inv_item_mapping = {v: k for k, v in item_mapping.items()}  # col  -> item_id

n_users = len(unique_users)
n_items = len(unique_items)
print(f"Ma tran R kich thuoc: {n_users} users x {n_items} items")
sparsity = 1 - len(df_inter) / (n_users * n_items)
print(f"Sparsity: {sparsity*100:.2f}%")

Ma tran R kich thuoc: 1000 users x 23929 items
Sparsity: 99.88%


In [5]:
# Anh xa sang chi so hang/cot
row_idx = df_inter["user_id"].map(user_mapping).values #Chi so dong
col_idx = df_inter["item_id"].map(item_mapping).values #Chi so cot
data    = df_inter["rating"].values.astype(np.float32)

# csr_matrix hieu qua cho TruncatedSVD.fit_transform
R = csr_matrix((data, (row_idx, col_idx)), shape=(n_users, n_items), dtype=np.float32)

print(f"R shape  : {R.shape}")
print(f"R nnz    : {R.nnz:,}  (so phan tu khac 0)")
print(f"R dtype  : {R.dtype}")

R shape  : (1000, 23929)
R nnz    : 29,785  (so phan tu khac 0)
R dtype  : float32


In [6]:
N_COMPONENTS = 50   # so chieu an
RANDOM_STATE = 42

print(f"Bat dau TruncatedSVD(n_components={N_COMPONENTS}) ...")
t0 = time.time()

#tao model SVD phan ra ma tran
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)

U_CF = svd.fit_transform(R).astype(np.float32)  # shape: (n_users, k)

V_CF = svd.components_.T.astype(np.float32)      # shape: (n_items, k)

elapsed = time.time() - t0
print(f"Hoan thanh trong {elapsed:.1f}s")
print(f"U_CF shape: {U_CF.shape}  (n_users x k)")
print(f"V_CF shape: {V_CF.shape}  (n_items x k)")

# Kiem tra explained variance
ev_ratio = svd.explained_variance_ratio_.sum()
print(f"Explained variance ratio (top-{N_COMPONENTS}): {ev_ratio*100:.2f}%")

Bat dau TruncatedSVD(n_components=50) ...
Hoan thanh trong 0.3s
U_CF shape: (1000, 50)  (n_users x k)
V_CF shape: (23929, 50)  (n_items x k)
Explained variance ratio (top-50): 8.38%


In [ ]:
def cf_recommend(user_id: int, top_k: int = 30) -> list[dict]:
    # User ngoai tap huan luyen duoc xu ly theo co che cold-start.
    if user_id not in user_mapping:
        return []   # cold-start
    if top_k <= 0:
        return []

    u_idx = user_mapping[user_id]
    p_u   = U_CF[u_idx] 

    # Tinh diem CF giua user va toan bo item bang tich vo huong.
    scores = (V_CF @ p_u).astype(np.float64)

    # Loai cac item da tuong tac de tranh goi y lap lai.
    seen_item_ids = set(df_inter.loc[df_inter["user_id"].eq(user_id), "item_id"].astype(int))
    if seen_item_ids:
        # Gan -inf de cac item nay khong duoc chon vao top-K.
        seen_indices = [item_mapping[iid] for iid in seen_item_ids if iid in item_mapping]
        scores[seen_indices] = -np.inf

    valid_count = int(np.isfinite(scores).sum())
    if valid_count == 0:
        return []

    top_k_eff = min(top_k, valid_count)

    # Trich xuat top-K item co diem CF cao nhat.
    top_indices = np.argpartition(scores, -top_k_eff)[-top_k_eff:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

    return [
        {"item_id": inv_item_mapping[idx], "cf_score": float(scores[idx])}
        for idx in top_indices
    ]


# Kiem tra nhanh ham CF voi mot user mau.
test_user = unique_users[0]
recs = cf_recommend(test_user, top_k=10)
print(f"CF top-10 cho user_id={test_user}:")
for r in recs:
    print(f"  item_id={r['item_id']}  cf_score={r['cf_score']:.4f}")

CF top-10 cho user_id=0:
  item_id=15501  cf_score=0.7876
  item_id=62650  cf_score=0.7206
  item_id=42516  cf_score=0.6725
  item_id=59772  cf_score=0.6718
  item_id=57456  cf_score=0.6376
  item_id=30490  cf_score=0.6169
  item_id=58345  cf_score=0.6025
  item_id=42130  cf_score=0.5941
  item_id=50546  cf_score=0.5829
  item_id=65740  cf_score=0.5828


In [ ]:
# Luu model pkl
cf_payload = {
    "U": U_CF,                          # (n_users, k) float32
    "V": V_CF,                          # (n_items, k) float32
    "svd": svd,                         # sklearn TruncatedSVD object
    "user_mapping": user_mapping,       # {user_id -> row_idx}
    "item_mapping": item_mapping,       # {item_id -> col_idx}
    "inv_user_mapping": inv_user_mapping,
    "inv_item_mapping": inv_item_mapping,
    "n_components": N_COMPONENTS,
    "n_users": n_users,
    "n_items": n_items,
}

pkl_path = OUTPUT_DIR / "cf_model.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(cf_payload, f, protocol=4)
print(f"Saved: {pkl_path}  ({pkl_path.stat().st_size/1e6:.2f} MB)")

# Luu mapping JSON (key phai la string trong JSON)
user_map_path = OUTPUT_DIR / "cf_user_mapping.json"
item_map_path = OUTPUT_DIR / "cf_item_mapping.json"

with open(user_map_path, "w") as f:
    json.dump({str(k): v for k, v in user_mapping.items()}, f)
print(f"Saved: {user_map_path}")

with open(item_map_path, "w") as f:
    json.dump({str(k): v for k, v in item_mapping.items()}, f)
print(f"Saved: {item_map_path}")

Saved: /kaggle/working/cf_model.pkl  (10.07 MB)
Saved: /kaggle/working/cf_user_mapping.json
Saved: /kaggle/working/cf_item_mapping.json


In [9]:
# Load FAISS index
print("Loading FAISS index ...")
faiss_index = faiss.read_index(str(INPUT_DIR / "fashion.index"))
print(f"  faiss_index: {faiss_index.ntotal:,} vectors")

# Load metadata (cung thu tu voi vector trong FAISS)
df_meta = pd.read_csv(INPUT_DIR / "metadata_indexed.csv")
df_meta["item_id"] = df_meta["item_id"].astype(int)
print(f"  metadata_indexed: {len(df_meta):,} rows")

# Load item_ids (anh xa faiss_idx -> item_id)
item_ids_arr = np.load(INPUT_DIR / "item_ids.npy")  # shape: (N,)
print(f"  item_ids_arr: {item_ids_arr.shape}")

assert len(item_ids_arr) == faiss_index.ntotal == len(df_meta), \
    "Do dai item_ids, FAISS index, metadata phai bang nhau!"

# Tao anh xa nhanh: faiss_idx -> item_id (mang) va item_id -> faiss_idx (dict)
faiss_idx_to_item_id = item_ids_arr.astype(int)           # (N,)
item_id_to_faiss_idx = {int(iid): idx for idx, iid in enumerate(item_ids_arr)}

# doc index_config de biet alpha hybrid da dung khi build index
with open(INPUT_DIR / "index_config.json") as f:
    idx_cfg = json.load(f)
ALPHA_INDEX = idx_cfg.get("alpha", 0.6)  # alpha dung khi tao combined_embeddings
print(f"  alpha (index build): {ALPHA_INDEX}")

print("\nLoad xong tat ca artifacts.")

Loading FAISS index ...
  faiss_index: 65,879 vectors
  metadata_indexed: 65,879 rows
  item_ids_arr: (65879,)
  alpha (index build): 0.6

Load xong tat ca artifacts.


In [10]:
!pip install git+https://github.com/openai/CLIP.git --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 

In [11]:
import torch
import torch.nn.functional as F
import clip
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

FINETUNE_CKPT = Path("/kaggle/input/notebooks/lehuuluongb2306557/fashion-dataset/checkpoints/fashion_clip_best.pt")

CLIP_MODEL_NAME = "ViT-B/32"
clip_model, clip_preprocess = clip.load(CLIP_MODEL_NAME, device=DEVICE, jit=False)
clip_model.eval()

if not FINETUNE_CKPT.exists():
    raise FileNotFoundError(
        f"Khong tim thay checkpoint fine-tuned: {FINETUNE_CKPT}. "
        "Hay add Kaggle Dataset chua fashion_clip_best.pt truoc khi chay"
    )

try:
    #Load checkpoint fine-tuned
    ckpt = torch.load(FINETUNE_CKPT, map_location=DEVICE, weights_only=False)
except TypeError:
    ckpt = torch.load(FINETUNE_CKPT, map_location=DEVICE)

#Nap trong so fine-tuned vao clip
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
clip_model.load_state_dict(state_dict)
clip_model.eval()
print(f"Da load fine-tuned checkpoint: {FINETUNE_CKPT}")


@torch.no_grad()
def encode_text_query(text: str) -> np.ndarray:
    #Encode mot cau text thanh vector L2-normalized (1, 512) float32.
    tokens = clip.tokenize([text], truncate=True).to(DEVICE)
    feat   = clip_model.encode_text(tokens)
    feat   = F.normalize(feat.float(), dim=-1)
    return feat.cpu().numpy().astype("float32")


@torch.no_grad()
def encode_image_query(image) -> np.ndarray:
    #Encode anh (PIL.Image hoac duong dan) thanh vector L2-normalized (1, 512) float32.
    if isinstance(image, (str, Path)):
        image = Image.open(image)
    img_t = clip_preprocess(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    feat  = clip_model.encode_image(img_t)
    feat  = F.normalize(feat.float(), dim=-1)
    return feat.cpu().numpy().astype("float32")


def make_query_vector(
    image=None,
    text: str | None = None,
    alpha: float = ALPHA_INDEX,
) -> np.ndarray:
    #TH1: Khong co anh va text
    if image is None and text is None:
        raise ValueError("Can it nhat mot trong hai: image hoac text.")

    #TH2: co anh va text
    if image is not None and text is not None:
        img_vec  = encode_image_query(image)   # (1, 512)
        txt_vec  = encode_text_query(text)     # (1, 512)
        combined = alpha * img_vec + (1 - alpha) * txt_vec
        norm     = np.linalg.norm(combined, axis=1, keepdims=True)
        return (combined / np.where(norm > 0, norm, 1.0)).astype("float32")

    #TH3: Chi co anh
    if image is not None:
        return encode_image_query(image)   # (1, 512)

    #TH4: Chi co text
    return encode_text_query(text)         # (1, 512)


print("Encode functions ready.")

Device: cuda


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 152MiB/s]


Da load fine-tuned checkpoint: /kaggle/input/notebooks/lehuuluongb2306557/fashion-dataset/checkpoints/fashion_clip_best.pt
Encode functions ready.


In [ ]:
def _minmax_norm(arr: np.ndarray, eps: float = 1e-9) -> tuple[np.ndarray, bool]:
    #Chuan hoa du lieu ve [0,1]
    lo, hi = arr.min(), arr.max()
    if hi - lo < eps:
        return np.zeros_like(arr, dtype=np.float32), True
    return ((arr - lo) / (hi - lo)).astype(np.float32), False


def _empty_hybrid_result() -> pd.DataFrame:
    #Tra ve DataFrame rong voi schema ket qua hybrid chuan.
    return pd.DataFrame(columns=[
        "rank", "item_id", "faiss_rank", "s_sem", "s_sem_norm",
        "cf_score", "c_ui_norm", "s_H", "has_cf",
        "category", "subcategory", "description",
    ])


def hybrid_recommend(
    query_image=None,
    query_text: str | None = None,
    user_id: int | None = None,
    top_k: int = 10,
    alpha_h: float = 0.6,
) -> pd.DataFrame:
    
    if top_k <= 0:
        raise ValueError("top_k phai lon hon 0")

    #Kiem tra cold-start
    is_cold_start = (user_id is None) or (user_id not in user_mapping)
    if is_cold_start:
        alpha_h = 1.0   # pure CLIP, bo qua CF hoan toan

    #FAISS search: lay ung vien va over-fetch neu can
    query_vec = make_query_vector(image=query_image, text=query_text)

    # Tap item da tuong tac dung de loc ket qua trung lap.
    seen_item_ids = set()
    if not is_cold_start:
        seen_item_ids = set(df_inter.loc[df_inter["user_id"].eq(user_id), "item_id"].astype(int))
    
    search_k = min(max(top_k * 3, top_k), faiss_index.ntotal)
    cand_item_ids = []
    sem_scores = np.array([], dtype=np.float64)
    faiss_idxs = np.array([], dtype=np.int64)

    while True:
        raw_scores, raw_indices = faiss_index.search(query_vec, search_k)  # (1, search_k) each
        current_scores = raw_scores[0].astype(np.float64)
        current_faiss_idxs = raw_indices[0]

        # FAISS co the tra -1 neu so ung vien yeu cau vuot qua index; loc an toan.
        valid_mask = current_faiss_idxs >= 0
        current_scores = current_scores[valid_mask]
        current_faiss_idxs = current_faiss_idxs[valid_mask]

        if len(current_faiss_idxs) == 0:
            return _empty_hybrid_result()

        current_item_ids = faiss_idx_to_item_id[current_faiss_idxs].astype(int).tolist()

        if seen_item_ids:
            # Loc item da rating truoc khi tinh CF/fusion.
            keep_mask = np.array([iid not in seen_item_ids for iid in current_item_ids], dtype=bool)
            current_scores = current_scores[keep_mask]
            current_faiss_idxs = current_faiss_idxs[keep_mask]
            current_item_ids = [iid for iid, keep in zip(current_item_ids, keep_mask) if keep]

        cand_item_ids = current_item_ids
        sem_scores = current_scores
        faiss_idxs = current_faiss_idxs

        # Neu loc xong van thieu ket qua, tang search_k va search lai.
        if len(cand_item_ids) >= top_k or search_k >= faiss_index.ntotal:
            break

        search_k = min(search_k * 2, faiss_index.ntotal)

    M = len(cand_item_ids)
    if M == 0:
        return _empty_hybrid_result()

    #CF scores cho tung ung vien (neu khong cold-start)
    cf_scores_raw = np.zeros(M, dtype=np.float64)  # mac dinh = 0
    has_cf_flag   = np.zeros(M, dtype=bool)        # item co trong I_CF khong

    if not is_cold_start:
        u_idx = user_mapping[user_id]
        p_u   = U_CF[u_idx].astype(np.float64)     # (k,)

        for pos, iid in enumerate(cand_item_ids):
            if iid in item_mapping:
                i_idx = item_mapping[iid]
                q_i   = V_CF[i_idx].astype(np.float64)  # (k,)
                cf_scores_raw[pos] = float(p_u @ q_i)   # c_ui
                has_cf_flag[pos]   = True

    #Chuan hoa min-max rieng tung nguon tren M ung vien
    s_sem_norm, sem_trivial = _minmax_norm(sem_scores)

    # CF norm chi tinh tren cac item CO cf score (has_cf=True)
    c_ui_norm = np.zeros(M, dtype=np.float32)
    if has_cf_flag.any():
        cf_subset = cf_scores_raw[has_cf_flag]
        cf_norm_subset, cf_trivial = _minmax_norm(cf_subset)
        c_ui_norm[has_cf_flag] = cf_norm_subset
    else:
        cf_trivial = True

    # Xu ly ngoai le: neu ca hai nguon deu trivial -> giu nguyen s_sem goc
    if sem_trivial and cf_trivial:
        s_sem_norm = (sem_scores / max(sem_scores.max(), 1e-9)).astype(np.float32)

    #Ket hop thanh s_H
    s_H = np.where(
        has_cf_flag,
        alpha_h * s_sem_norm + (1.0 - alpha_h) * c_ui_norm,  # co CF
        s_sem_norm,                                          # khong co CF -> dung s_sem
    ).astype(np.float32)

    # Sap xep giam dan theo s_H, lay top-K
    ranked_pos = np.argsort(s_H)[::-1][:top_k]

    result_rows = []
    for rank, pos in enumerate(ranked_pos):
        iid = cand_item_ids[pos]
        result_rows.append({
            "rank"       : rank + 1,
            "item_id"    : iid,
            "faiss_rank" : int(pos + 1),
            "s_sem"      : float(sem_scores[pos]),
            "s_sem_norm" : float(s_sem_norm[pos]),
            "cf_score"   : float(cf_scores_raw[pos]),
            "c_ui_norm"  : float(c_ui_norm[pos]),
            "s_H"        : float(s_H[pos]),
            "has_cf"     : bool(has_cf_flag[pos]),
        })

    df_result = pd.DataFrame(result_rows)

    # Join them thong tin metadata
    df_result = df_result.merge(
        df_meta[["item_id", "category", "subcategory", "description"]]
              .astype({"item_id": int}),
        on="item_id",
        how="left",
    )

    return df_result


print("hybrid_recommend() da san sang.")

hybrid_recommend() da san sang.


In [13]:
SAMPLE_QUERY = "black floral dress"
SAMPLE_USER  = unique_users[0]   # user_id=0 hoac user dau tien co trong CF

print(f"=== QUERY: '{SAMPLE_QUERY}' | USER: {SAMPLE_USER} ===")

# --- Pure CLIP (alpha_h=1.0) ---
df_clip = hybrid_recommend(
    query_text=SAMPLE_QUERY,
    user_id=None,   # cold-start -> pure CLIP
    top_k=10,
    alpha_h=0.6,
)
print("\n[Pure CLIP - cold-start]")
print(df_clip[["rank", "item_id", "s_sem", "s_H", "category", "description"]]
      .assign(description=df_clip["description"].str[:60])
      .to_string(index=False))

# --- Hybrid (alpha_h=0.6) ---
df_hybrid = hybrid_recommend(
    query_text=SAMPLE_QUERY,
    user_id=SAMPLE_USER,
    top_k=10,
    alpha_h=0.6,
)
print(f"\n[Hybrid CLIP+CF (alpha_h=0.6) - user {SAMPLE_USER}]")
print(df_hybrid[["rank", "item_id", "s_sem_norm", "c_ui_norm", "s_H", "has_cf", "description"]]
      .assign(description=df_hybrid["description"].str[:55])
      .to_string(index=False))

=== QUERY: 'black floral dress' | USER: 0 ===

[Pure CLIP - cold-start]
 rank  item_id    s_sem      s_H category                   description
    1    29653 0.729903 1.000000  dresses            black floral dress
    2    28374 0.729493 0.990427  dresses            black floral dress
    3    29613 0.722022 0.816136  dresses            black floral dress
    4    16197 0.719916 0.766996  dresses      black floral print dress
    5    21898 0.718767 0.740196  dresses      black floral print dress
    6    36553 0.718450 0.732806  dresses      black floral-print dress
    7    34309 0.712480 0.593512  dresses multicolor floral print dress
    8    24892 0.708100 0.491330  dresses multicolor floral print dress
    9    31421 0.705197 0.423614  dresses multicolor floral print dress
   10    20842 0.699925 0.300609  dresses multicolor floral print dress

[Hybrid CLIP+CF (alpha_h=0.6) - user 0]
 rank  item_id  s_sem_norm  c_ui_norm      s_H  has_cf                   description
    1    2

In [14]:
# --- Kiem tra CF co THUC SU tai xep hang khong ---
clip_item_order   = df_clip["item_id"].tolist()
hybrid_item_order = df_hybrid["item_id"].tolist()

changed = sum(1 for a, b in zip(clip_item_order, hybrid_item_order) if a != b)
print(f"So vi tri thay doi giua pure-CLIP va Hybrid: {changed}/10")

if changed == 0:
    print("[CANH BAO] Thu tu hoan toan giong nhau.")
    print("  -> Kiem tra interactions.csv: user nay co the khong co rating cho cac item trong top-30?")
else:
    print("OK: CF da tai xep hang ket qua.")

So vi tri thay doi giua pure-CLIP va Hybrid: 8/10
OK: CF da tai xep hang ket qua.


In [15]:
# Chon mot user_id chac chan khong co trong CF
NON_EXIST_USER = -9999
assert NON_EXIST_USER not in user_mapping, "Chon user_id khac (khong ton tai trong CF)"

df_cold = hybrid_recommend(
    query_text=SAMPLE_QUERY,
    user_id=NON_EXIST_USER,
    top_k=10,
    alpha_h=0.6,
)

print(f"Cold-start (user_id={NON_EXIST_USER}):")
print(df_cold[["rank", "item_id", "s_sem_norm", "c_ui_norm", "s_H", "has_cf"]].to_string(index=False))

# Xac nhan: c_ui_norm phai = 0 cho tat ca, s_H phai = s_sem_norm
assert (df_cold["c_ui_norm"] == 0.0).all(), "c_ui_norm phai = 0 khi cold-start!"
assert (df_cold["has_cf"] == False).all(), "has_cf phai = False khi cold-start!"
assert np.allclose(df_cold["s_H"].values, df_cold["s_sem_norm"].values, atol=1e-5), \
    "s_H phai == s_sem_norm khi alpha_h=1.0!"
print("\nPass: Cold-start fallback hoat dong dung.")

Cold-start (user_id=-9999):
 rank  item_id  s_sem_norm  c_ui_norm      s_H  has_cf
    1    29653    1.000000        0.0 1.000000   False
    2    28374    0.990427        0.0 0.990427   False
    3    29613    0.816136        0.0 0.816136   False
    4    16197    0.766996        0.0 0.766996   False
    5    21898    0.740196        0.0 0.740196   False
    6    36553    0.732806        0.0 0.732806   False
    7    34309    0.593512        0.0 0.593512   False
    8    24892    0.491330        0.0 0.491330   False
    9    31421    0.423614        0.0 0.423614   False
   10    20842    0.300609        0.0 0.300609   False

Pass: Cold-start fallback hoat dong dung.


In [16]:
# Lay ket qua hybrid voi user hop le
df_mixed = hybrid_recommend(
    query_text=SAMPLE_QUERY,
    user_id=SAMPLE_USER,
    top_k=10,
    alpha_h=0.6,
)

items_outside_cf = df_mixed[df_mixed["has_cf"] == False]
print(f"So item trong top-10 KHONG co CF score: {len(items_outside_cf)}")
if not items_outside_cf.empty:
    print("Cac item nay van xuat hien trong ket qua (khong bi loai):")
    print(items_outside_cf[["rank", "item_id", "s_sem_norm", "c_ui_norm", "s_H"]]
          .to_string(index=False))
    # Xac nhan s_H = s_sem_norm voi item khong co CF
    assert np.allclose(
        items_outside_cf["s_H"].values,
        items_outside_cf["s_sem_norm"].values,
        atol=1e-5,
    ), "Item ngoai I_CF phai co s_H = s_sem_norm!"
    print("Pass: item ngoai I_CF van giu s_H = s_sem_norm.")
else:
    print("(Tat ca top-10 deu co CF score - OK trong truong hop interactions day du.)") 

So item trong top-10 KHONG co CF score: 5
Cac item nay van xuat hien trong ket qua (khong bi loai):
 rank  item_id  s_sem_norm  c_ui_norm      s_H
    1    29653    1.000000        0.0 1.000000
    2    29613    0.816136        0.0 0.816136
    3    16197    0.766996        0.0 0.766996
    4    21898    0.740196        0.0 0.740196
   10    31421    0.423614        0.0 0.423614
Pass: item ngoai I_CF van giu s_H = s_sem_norm.


In [17]:
N_BENCH = 20
times = []
for _ in range(N_BENCH):
    t0 = time.time()
    _ = hybrid_recommend(
        query_text=SAMPLE_QUERY,
        user_id=SAMPLE_USER,
        top_k=10,
        alpha_h=0.6,
    )
    times.append((time.time() - t0) * 1000)

print(f"Thoi gian hybrid_recommend() qua {N_BENCH} lan:")
print(f"  Trung binh : {np.mean(times):.1f} ms")
print(f"  P50        : {np.percentile(times, 50):.1f} ms")
print(f"  P95        : {np.percentile(times, 95):.1f} ms")
print(f"  Max        : {np.max(times):.1f} ms")

target_ms = 3000
if np.mean(times) < target_ms:
    print(f"OK: Dat muc tieu < {target_ms} ms/query.")
else:
    print(f"CANH BAO: Chua dat muc tieu {target_ms} ms, kiem tra lai cache model.")

Thoi gian hybrid_recommend() qua 20 lan:
  Trung binh : 37.3 ms
  P50        : 37.2 ms
  P95        : 38.6 ms
  Max        : 41.3 ms
OK: Dat muc tieu < 3000 ms/query.


In [ ]:
ARTIFACTS_W6 = [
    "cf_model.pkl",
    "cf_user_mapping.json",
    "cf_item_mapping.json",
]

print("=== Artifacts cuoi Tuan 6 ===")
all_ok = True
for fname in ARTIFACTS_W6:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        print(f"  [OK] {fname:40s}  {size_mb:.2f} MB")
    else:
        print(f"  [MISSING] {fname}")
        all_ok = False

=== Artifacts cuoi Tuan 6 ===
  [OK] cf_model.pkl                              10.07 MB
  [OK] cf_user_mapping.json                      0.01 MB
  [OK] cf_item_mapping.json                      0.37 MB

Tat ca artifacts da duoc luu. San sang upload len Kaggle Dataset cho Tuan 7.


In [19]:
# Ket qua test cuoi
milestones = [
    ("cf_model.pkl hoat dong, goi y dung top-K",
     (OUTPUT_DIR / "cf_model.pkl").exists()),
    ("Hybrid fusion tra ve ket qua ket hop CLIP + CF",
     len(df_hybrid) == 10),
    ("Cold-start fallback ve alpha=1.0 (pure CLIP)",
     True),   # da assert o tren
    ("Item ngoai I_CF van giu s_H = s_sem_norm",
     True),   # da assert o tren
]
for desc, status in milestones:
    icon = "[PASS]" if status else "[FAIL]"
    print(f"  {icon}  {desc}")

  [PASS]  cf_model.pkl hoat dong, goi y dung top-K
  [PASS]  Hybrid fusion tra ve ket qua ket hop CLIP + CF
  [PASS]  Cold-start fallback ve alpha=1.0 (pure CLIP)
  [PASS]  Item ngoai I_CF van giu s_H = s_sem_norm
